# Block 14 — Ensembles: Heterogeneous Models & Evaluating the Whole Arc
### Advanced Machine Learning — M&T Bank

**Dataset:** `bank_marketing_features.csv` (unchanged since Day 1, Block 3). Same target/split. No new CSV comes
out of this block.

Blocks 12-13 combined many copies of the *same* model type (bagging, random forest, boosting) — all homogeneous
ensembles. Today: combine *different* model types, and close out the challenger-model exercise with a verdict.


## Setup

In [ ]:
import warnings
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

# Paste the raw GitHub URL for bank_marketing_features.csv below, then remove the leading '#':
# df = pd.read_csv("PASTE_RAW_GITHUB_URL_HERE", sep=";")
feature_cols = [c for c in df.columns if c not in ("education", "y", "duration")]
bool_cols = [c for c in df[feature_cols].columns if df[c].dtype == bool]
X = df[feature_cols].copy()
for c in bool_cols:
    X[c] = X[c].astype(int)
y = df["y"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_to_scale = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
    "education_rank", "month_sin", "month_cos", "campaign_intensity",
    "emp.var.rate_denoised", "cons.price.idx_denoised", "cons.conf.idx_denoised",
    "euribor3m_denoised", "nr.employed_denoised",
]
# Scaling lives inside a ColumnTransformer/Pipeline so it stays leakage-safe even inside
# VotingClassifier/StackingClassifier's internal cross-validation (Block 5/10's rule, still applies).
ct = ColumnTransformer([("scale", StandardScaler(), numeric_to_scale)], remainder="passthrough")
lr_pipe = Pipeline([("prep", ct), ("clf", LogisticRegression(max_iter=2000, C=0.5))])
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
gbt = HistGradientBoostingClassifier(random_state=42)


## 1. The Three Individual Models, Reconfirmed


In [2]:
lr_pipe.fit(Xtr, ytr)
lr_auc = roc_auc_score(yte, lr_pipe.predict_proba(Xte)[:, 1])
rf.fit(Xtr, ytr)
rf_auc = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])
gbt.fit(Xtr, ytr)
gbt_auc = roc_auc_score(yte, gbt.predict_proba(Xte)[:, 1])

pd.DataFrame({
    "Model": ["Logistic regression (Block 6)", "Random Forest (max_depth=10, Block 12)", "Gradient Boosting (Block 12)"],
    "Test ROC-AUC": [lr_auc, rf_auc, gbt_auc],
}).round(4)


,Model,Test ROC-AUC
0,Logistic regression (Block 6),0.7958
1,"Random Forest (max_depth=10, Block 12)",0.8161
2,Gradient Boosting (Block 12),0.8180


## 2. Combining Different Model Types

**Voting:** average the predicted probabilities of several different, already-fit models.
**Stacking:** train a small "meta-model" (here, another logistic regression) whose job is to learn how to best
combine the base models' predictions, using cross-validated out-of-fold predictions to avoid leaking training
performance into the meta-model.


In [3]:
t0 = time.time()
voting = VotingClassifier(estimators=[("lr", lr_pipe), ("rf", rf), ("gbt", gbt)], voting="soft", n_jobs=-1)
voting.fit(Xtr, ytr)
voting_auc = roc_auc_score(yte, voting.predict_proba(Xte)[:, 1])
print(f"Voting (soft, equal weight, LR+RF+GBT): test AUC = {voting_auc:.4f}  ({time.time()-t0:.1f}s)")

t0 = time.time()
stack = StackingClassifier(
    estimators=[("lr", lr_pipe), ("rf", rf), ("gbt", gbt)],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5, n_jobs=-1,
)
stack.fit(Xtr, ytr)
stack_auc = roc_auc_score(yte, stack.predict_proba(Xte)[:, 1])
print(f"Stacking (5-fold, LR meta-learner, LR+RF+GBT): test AUC = {stack_auc:.4f}  ({time.time()-t0:.1f}s)")


Voting (soft, equal weight, LR+RF+GBT): test AUC = 0.8152  (4.6s)


Stacking (5-fold, LR meta-learner, LR+RF+GBT): test AUC = 0.8179  (11.9s)


## 3. Does Dropping the Weakest Model Help?

The logistic regression is the weakest of the three individual models (0.796 vs. 0.816/0.818). Two more attempts:
weight the vote toward the stronger models, and try voting with just the two tree ensembles.


In [4]:
voting_weighted = VotingClassifier(estimators=[("lr", lr_pipe), ("rf", rf), ("gbt", gbt)],
                                    voting="soft", weights=[1, 2, 3], n_jobs=-1)
voting_weighted.fit(Xtr, ytr)
voting_w_auc = roc_auc_score(yte, voting_weighted.predict_proba(Xte)[:, 1])

voting_rf_gbt = VotingClassifier(estimators=[("rf", rf), ("gbt", gbt)], voting="soft", n_jobs=-1)
voting_rf_gbt.fit(Xtr, ytr)
voting_rf_gbt_auc = roc_auc_score(yte, voting_rf_gbt.predict_proba(Xte)[:, 1])

pd.DataFrame({
    "Ensemble": ["Voting, equal weight (LR+RF+GBT)", "Stacking (LR+RF+GBT)",
                 "Voting, weighted 1/2/3 (LR+RF+GBT)", "Voting, equal weight (RF+GBT only)"],
    "Test ROC-AUC": [voting_auc, stack_auc, voting_w_auc, voting_rf_gbt_auc],
}).round(4)


,Ensemble,Test ROC-AUC
0,"Voting, equal weight (LR+RF+GBT)",0.8152
1,Stacking (LR+RF+GBT),0.8179
2,"Voting, weighted 1/2/3 (LR+RF+GBT)",0.8174
3,"Voting, equal weight (RF+GBT only)",0.8184


## 4. The Full Scoreboard — Does Combining Models Beat the Best Single One?


In [5]:
pd.DataFrame({
    "Model": ["Logistic regression", "Random Forest (tuned)", "Gradient Boosting (best single model)",
              "Voting (LR+RF+GBT)", "Stacking (LR+RF+GBT)", "Voting weighted (1/2/3)", "Voting (RF+GBT only)"],
    "Test ROC-AUC": [lr_auc, rf_auc, gbt_auc, voting_auc, stack_auc, voting_w_auc, voting_rf_gbt_auc],
}).round(4).sort_values("Test ROC-AUC", ascending=False).reset_index(drop=True)


,Model,Test ROC-AUC
0,Voting (RF+GBT only),0.8184
1,Gradient Boosting (best single model),0.8180
2,Stacking (LR+RF+GBT),0.8179
3,Voting weighted (1/2/3),0.8174
4,Random Forest (tuned),0.8161
5,Voting (LR+RF+GBT),0.8152
6,Logistic regression,0.7958


**Every combination attempt lands within 0.001-0.003 of Gradient Boosting alone (0.818) — none of them clear it by
a margin worth the added complexity of maintaining three models instead of one.** Equal-weight voting with the
logistic regression included (0.815) was actually the worst of the group, dragged down by averaging in the
weakest model. Dropping it or weighting around it (0.817-0.818) closed most of that gap, but never opened a real
lead over GBT by itself.


## 5. Closing the Ensembles Arc

Three blocks, one challenger-model question asked three different ways.


In [6]:
arc_summary = pd.DataFrame([
    ["Block 12", "RF vs. GBT vs. LR benchmark", "Untuned RF lost to the benchmark; tuned RF and GBT both beat it (0.816 / 0.818 vs. 0.796)"],
    ["Block 13", "Why bagging works", "Two bootstrapped trees agreed on only 84.8% of clients; averaging 100 of them turned AUC 0.62 into 0.77"],
    ["Block 14", "Combining different model types", "Voting/stacking all landed within noise of GBT alone (0.818) — the extra complexity didn't pay for itself"],
], columns=["Block", "Question asked", "What we found"])
arc_summary


,Block,Question asked,What we found
0,Block 12,RF vs. GBT vs. LR benchmark,Untuned RF lost to the benchmark; tuned RF and...
1,Block 13,Why bagging works,Two bootstrapped trees agreed on only 84.8% of...
2,Block 14,Combining different model types,Voting/stacking all landed within noise of GBT...


**The challenger-model verdict for this dataset: Gradient Boosting, on its own, is the model that earned its
deployment.** Not because it's the most sophisticated-sounding option on the table — heterogeneous ensembling is
arguably the more sophisticated-sounding choice — but because everything more complex than it either lost to it
outright (default Random Forest) or failed to clear it by a margin worth the added maintenance burden (voting,
stacking). That's the entire discipline in one dataset: complexity is a cost, and it has to be paid for with a
real, measured improvement — not assumed.
